# Doğal Dil İşleme + Karar Bilimi (Natural Language Processing + Decision Science)

👩🏻‍🏫 Bu görevde şunları bir araya getireceğiz:
* 🗣 Doğal Dil İşleme (Natural Language Processing)
* 📊 Karar Bilimi (Decision Science)

🎯 Amaç, Olist üzerindeki ürünlerin ve satıcıların **olumsuz (kötü) yorumlarını** anlamaktır.

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
# Data Manipulation
import numpy as np
import pandas as pd
pd.set_option("display.max_columns",None)

# Machine Learning
from sklearn.pipeline import make_pipeline

# Language Processing
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
import string
import unidecode as unidecode

# Vectorizers and NLP Models
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

🕵🏻‍♂️ Olist’in CEO’su [Tiago Dalvi](https://www.linkedin.com/in/tiagodalvi/)’nin senden yorumları okuyup anlamanı istediğini hayal et.

- Müşteriler siparişlerini **1**, **2** veya **3** puanla değerlendirdiklerinde ne söylediler?
- En sık karşılaşılan olumsuz yorumlar neler?
    - En kötü puanlanan ürünler hakkında?
    - En kötü puanlanan satıcılar hakkında?


## (0) Kurulum 🔨

Öncelikle, Olist incelemeleriyle ilgili tüm bilgileri içeren DataFrame'i yükleyeceğiz!

In [5]:
df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/olist_reviews.csv")

In [6]:
df.head()

,review_id,length_review,review_score,order_id,product_category_name,full_review
0,e64fb393e7b32834bb789ff8bb30750e,37,5,658677c97b385a9be170737859d3511b,ferramentas_jardim,Recebi bem antes do prazo estipulado.
1,f7c4243c7fe1938f181bec41a392bdeb,100,5,8e6bfb81e283fa7e4f11123a3fb894f1,esporte_lazer,Parabéns lojas lannister adorei comprar pela ...
2,8670d52e15e00043ae7de4c01cc2fe06,174,4,b9bf720beb4ab3728760088589c62129,eletroportateis,recomendo aparelho eficiente. no site a marca ...
3,4b49719c8a200003f700d3d986ea1a19,45,4,9d6f15f95d01e79bd1349cc208361f09,beleza_saude,"Mas um pouco ,travando...pelo valor ta Boa.\r\n"
4,3948b09f7c818e2d86c9a546758b2335,56,5,e51478e7e277a83743b6f9991dbfa3fb,informatica_acessorios,"Super recomendo Vendedor confiável, produto ok..."


## (2) Metin Temizleme (Text Cleaning)

🧹 `cleaning(sentence)` işlevini oluşturun ve yorumlara uygulayın. **NLTK'da Portekizce lemmatizer bulunmadığını unutmayın** (genellikle bulunmaz, ancak `nltk.stem.RSLPStemmer` kök bulucu vardır).

In [7]:
for pkg in ["stopwords", "punkt", "punkt_tab"]:
    nltk.download(pkg, quiet=True)

stop_words_pt = set(stopwords.words("portuguese"))

def cleaning(sentence):
    sentence = str(sentence).strip().lower()                          
    sentence = "".join(ch for ch in sentence if not ch.isdigit())        
    for p in string.punctuation:                                          
        sentence = sentence.replace(p, " ")
    sentence = unidecode.unidecode(sentence)                              
    tokens = word_tokenize(sentence)
    tokens = [w for w in tokens if w not in stop_words_pt and len(w) > 1]
    return " ".join(tokens)

cleaning("O pedido 45 NÃO foi entregue ainda! Péssimo serviço.")

'pedido nao entregue ainda pessimo servico'

In [8]:
if "full_review" in df.columns:
    full_review = df["full_review"].fillna("")
else:
    title_col = [c for c in df.columns if "title" in c]
    msg_col = [c for c in df.columns if "message" in c]
    full_review = df[title_col[0]].fillna("") + " " + df[msg_col[0]].fillna("")

df["full_review_cleaned"] = full_review.apply(cleaning)
df = df[df["full_review_cleaned"].str.len() > 0].reset_index(drop=True)   # boş yorumları at
df[["review_score", "full_review_cleaned"]].head()

,review_score,full_review_cleaned
0,5,recebi bem antes prazo estipulado
1,5,parabens lojas lannister adorei comprar intern...
2,4,recomendo aparelho eficiente site marca aparel...
3,4,pouco travando valor ta boa
4,5,super recomendo vendedor confiavel produto ok ...


## (3) Kötü yorumların analizi

### (3.1) Düşük inceleme puanlarına sahip veri kümesi

😱 1 ile 3 arasında puan alan yorumların oranı nedir? 

In [9]:
bad_ratio = (df["review_score"] <= 3).mean()
print(f"1-3 puanlı yorumların oranı: {bad_ratio:.1%}")

1-3 puanlı yorumların oranı: 26.8%


🕵🏻‍♂️ Bu yorumlara odaklanalım...

In [10]:
df = df[df["review_score"] <= 3].reset_index(drop=True)
print(df.shape)
df["review_score"].value_counts().sort_index()

(9627, 7)


review_score
1    4852
2    1656
3    3119
Name: count, dtype: int64

### (3.2) Vektörleştirme

🔡 ➡️ 🔢 Metinlerini vektörleştir.

- **Bigram**’leri (iki kelimelik ifadeler) mutlaka hesaba kat.
- Çok sık geçen kelimeleri çıkarmak için `max_df = 0.75` ayarla.
- Spoiler uyarısı: Sonunda **20.000+** kelimeye ulaşacaksın…  
  Bu challenge için sadece `max_features = 5000` ile sınırla.

In [11]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_df=0.75, max_features=5000)
X_vectorized = vectorizer.fit_transform(df["full_review_cleaned"])
X_vectorized.shape

(9627, 5000)

### (3.3) LDA

🕵🏻‍♂️ LDA'yı uyarlayın:
- `n_components = 3` seçin
- *.transform()* ile Konuların Belge Karışımını gösterin
- *.components_* ile Konu Karışımını gösterin

In [12]:
lda = LatentDirichletAllocation(n_components=3, random_state=42)
lda.fit(X_vectorized)

LatentDirichletAllocation(n_components=3, random_state=42)

#### Belge Karışımı (Konular için)

In [13]:
document_mixture = lda.transform(X_vectorized)
pd.DataFrame(document_mixture, columns=["topic_0", "topic_1", "topic_2"]).round(3).head()

,topic_0,topic_1,topic_2
0,0.085,0.219,0.696
1,0.076,0.848,0.076
2,0.842,0.074,0.084
3,0.101,0.101,0.798
4,0.742,0.129,0.129


👉 Her inceleme için en önemli konuyu rapor edelim

In [14]:
df["most_important_topic"] = document_mixture.argmax(axis=1)
df["most_important_topic"].value_counts()

most_important_topic
1    3471
0    3352
2    2804
Name: count, dtype: int64

#### Konu Karışımı (Kelimeler için)

In [15]:
topic_mixture = pd.DataFrame(lda.components_, columns=vectorizer.get_feature_names_out())
topic_mixture.round(2).iloc[:, :15]

,abaixo,aberta,aberto,abri,abri caixa,abri embalagem,abri reclamacao,abrindo,abrir,abrir embalagem,absurdo,acabada,acabado,acabamento,acabamento nao
0,0.61,4.19,0.55,1.95,1.08,1.00,0.35,0.48,1.69,0.44,5.37,0.36,0.34,7.89,1.02
1,0.36,1.21,0.41,9.05,0.86,1.31,4.45,2.28,4.10,1.67,4.08,0.34,0.33,0.42,0.75
2,3.47,2.39,7.88,1.34,0.34,0.72,0.35,0.37,7.04,0.68,3.47,2.08,5.59,15.22,0.75


#### Konular

🎁 Size bazı yardımcı fonksiyonlar sağladık:
- `topic_word`: Tek bir konu (topic) için en önemli kelimeleri ve ağırlıklarını döndürür
- `print_topics`: LDA tarafından bulunan farklı konuları, en önemli kelimeleriyle birlikte yazdırır

In [16]:
def topic_word(vectorizer, model, topic, topwords, with_weights = True):
    topwords_indexes = topic.argsort()[:-topwords - 1:-1]
    if with_weights == True:
        topwords = [(vectorizer.get_feature_names_out()[i], round(topic[i],2)) for i in topwords_indexes]
    if with_weights == False:
        topwords = [vectorizer.get_feature_names_out()[i] for i in topwords_indexes]
    return topwords

In [17]:
def print_topics(vectorizer, model, topwords):
    for idx, topic in enumerate(model.components_):
        print("-"*20)
        print("Topic %d:" % (idx))
        print(topic_word(vectorizer, model, topic, topwords))


🕵🏻‍♂️ Konuları en çok kullanılan kelimelerle birlikte yazdırın:

In [18]:
print_topics(vectorizer, lda, topwords=10)

--------------------
Topic 0:
[('nao', 240.85), ('bom', 229.56), ('produto', 193.03), ('entrega', 108.35), ('gostei', 93.87), ('prazo', 85.42), ('nao recebi', 85.27), ('recebi', 84.54), ('ainda', 80.14), ('recebi produto', 69.87)]
--------------------
Topic 1:
[('recebi', 209.49), ('comprei', 156.64), ('nao', 144.1), ('so', 137.83), ('entregue', 133.43), ('apenas', 106.21), ('produto', 94.32), ('pedido', 91.66), ('chegou', 90.81), ('dois', 88.11)]
--------------------
Topic 2:
[('produto', 203.17), ('veio', 120.23), ('nao', 110.71), ('recomendo', 81.6), ('diferente', 68.88), ('produto veio', 65.52), ('qualidade', 60.23), ('errado', 60.22), ('bem', 54.8), ('ruim', 54.06)]


🇧🇷 Burada biraz Brezilya Portekizcesi kelimeler var:
- _cadeiras = chairs_
- _produto = product_
- _recomendo = recommend (não recomendo == not recommend)_
- _bom = good_
- _comprei = bought_
- _veio = came_
- _errado = wrong_
- _gostaria = I would like to..._
- _duas = two_
- _nao = not_
- _entregue = delivered_
- _pecas = part_
- _ainda = yet_
- _recebi = received_

👉 Bir konuyla ilişkili en popüler kelimeleri göster

In [19]:
topic_word_mixture = [topic_word(vectorizer, lda, topic, 10, with_weights=False) for topic in lda.components_]
topic_word_mixture

[['nao',
  'bom',
  'produto',
  'entrega',
  'gostei',
  'prazo',
  'nao recebi',
  'recebi',
  'ainda',
  'recebi produto'],
 ['recebi',
  'comprei',
  'nao',
  'so',
  'entregue',
  'apenas',
  'produto',
  'pedido',
  'chegou',
  'dois'],
 ['produto',
  'veio',
  'nao',
  'recomendo',
  'diferente',
  'produto veio',
  'qualidade',
  'errado',
  'bem',
  'ruim']]

In [20]:
df["most_important_words"] = df["most_important_topic"].apply(lambda i: topic_word_mixture[i])

In [21]:
df[["review_id",
        "review_score",
        "product_category_name",
        "full_review_cleaned",
        "most_important_topic",
        "most_important_words"]
      ].head()

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
0,e233e51d11511bf30e568c76360ace52,1,eletronicos,recebi somente controle midea split estilo fal...,2,"[produto, veio, nao, recomendo, diferente, pro..."
1,8b230a1373c6dc4bd867099fda1d7039,3,ferramentas_jardim,comprei duas unidades so recebi agora faco,1,"[recebi, comprei, nao, so, entregue, apenas, p..."
2,cb2fc3e5711b5ae85e0491ee18af63ed,3,beleza_saude,produto bom porem veio mim nao condiz foto anu...,0,"[nao, bom, produto, entrega, gostei, prazo, na..."
3,60c714ed14cef913944a3147094a4742,1,moveis_decoracao,produto inferior mal acabado,2,"[produto, veio, nao, recomendo, diferente, pro..."
4,0bd4dcc4f6c4621baf37f73495cad8c4,3,esporte_lazer,entrega prazo,0,"[nao, bom, produto, entrega, gostei, prazo, na..."


## (3.4) Pipeline Tf-Idf ve LDA

In [22]:
from sklearn import set_config
set_config("diagram")

🔨 Önceki Vectorizer ve LDA'yı birbirine bağlayan bir Pipeline oluşturun.

Temizlenmiş metinlere uyarlayın.

In [23]:
pipeline = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), max_df=0.75, max_features=5000),
    LatentDirichletAllocation(n_components=3, random_state=42),
)
pipeline.fit(df["full_review_cleaned"])

Pipeline(steps=[('tfidfvectorizer',
                 TfidfVectorizer(max_df=0.75, max_features=5000,
                                 ngram_range=(1, 2))),
                ('latentdirichletallocation',
                 LatentDirichletAllocation(n_components=3, random_state=42))])

💡 `pipeline.components_` ile bileşenlere erişmeye çalışırsanız, Pipeline'da `components_` olmadığı için bu YÜRÜMEZ. Ancak, LDA'ya erişmek için `pipeline._final_estimator` kullanabilirsiniz. Ve buradan konulara erişebilirsiniz!

In [24]:
pipeline._final_estimator

LatentDirichletAllocation(n_components=3, random_state=42)

In [25]:
pipeline._final_estimator.components_

array([[0.61342353, 4.19342361, 0.54768369, ..., 1.17100573, 1.08029913,
        6.16345689],
       [0.36344388, 1.21281155, 0.41160678, ..., 0.34066002, 6.46634757,
        0.72725416],
       [3.46704856, 2.39043648, 7.88443885, ..., 2.01578157, 1.35358684,
        0.40628213]])

Pipeline ile **Belge Karışımı**:

In [26]:
pd.DataFrame(pipeline.transform(df["full_review_cleaned"]), columns=["topic_0", "topic_1", "topic_2"]).round(3).head()

,topic_0,topic_1,topic_2
0,0.085,0.219,0.696
1,0.076,0.848,0.076
2,0.842,0.074,0.084
3,0.101,0.101,0.798
4,0.742,0.129,0.129


Pipeline ile **Konu Karışımı**:

In [27]:
pd.DataFrame(pipeline._final_estimator.components_,
             columns=pipeline[0].get_feature_names_out()).round(2).iloc[:, :15]

,abaixo,aberta,aberto,abri,abri caixa,abri embalagem,abri reclamacao,abrindo,abrir,abrir embalagem,absurdo,acabada,acabado,acabamento,acabamento nao
0,0.61,4.19,0.55,1.95,1.08,1.00,0.35,0.48,1.69,0.44,5.37,0.36,0.34,7.89,1.02
1,0.36,1.21,0.41,9.05,0.86,1.31,4.45,2.28,4.10,1.67,4.08,0.34,0.33,0.42,0.75
2,3.47,2.39,7.88,1.34,0.34,0.72,0.35,0.37,7.04,0.68,3.47,2.08,5.59,15.22,0.75


## (4) 🎁 Ürün Kategorileri

### (4.1) Ürün kategorilerine göre gruplandırma

📈 Veri kümesini `product_category_name` ile gruplandırın ve performanslarını inceleyin.

In [28]:
# Performansa göre ürün kategorileri - sayı, ortalama, medyan ve standart sapmaya bakın
product_categories = df.groupby(by = 'product_category_name').agg({
        'review_score': ["count", "mean", "median", "std"]
    })

# Analiz için belirli bir süreden daha az satılan ürünleri kaldırma
cutoff = 50
product_categories = product_categories[product_categories[("review_score", "count")] > cutoff]

# Ürün kategorilerini performansa göre sıralama
product_categories = product_categories.sort_values(by = [('review_score', 'mean'),
                                                          ('review_score', 'std')],
                                                    ascending = [False, True])
product_categories

review_score                           
                                         count      mean median       std
product_category_name                                                    
consoles_games                              92  1.956522    2.0  0.924787
fashion_bolsas_e_acessorios                155  1.948387    2.0  0.917319
malas_acessorios                            66  1.924242    2.0  0.949666
papelaria                                  155  1.922581    2.0  0.922556
casa_construcao                             65  1.907692    2.0  0.842672
utilidades_domesticas                      583  1.886792    2.0  0.887609
cool_stuff                                 298  1.885906    2.0  0.906618
relogios_presentes                         586  1.878840    2.0  0.904308
pet_shop                                   147  1.857143    2.0  0.883641
moveis_escritorio                          261  1.854406    1.0  0.920843
cama_mesa_banho                           1213  1.846661    2.0  0.888819
esporte_lazer                              624  1.844551    2.0  0.899368
casa_conforto                               57  1.842105    2.0  0.902169
beleza_saude                               715  1.839161    1.0  0.907874
ferramentas_jardim                         318  1.817610    1.0  0.904635
telefonia                                  488  1.815574    2.0  0.875636
moveis_decoracao                           740  1.800000    1.0  0.887745
brinquedos                                 298  1.788591    1.0  0.890808
bebes                                      244  1.786885    1.0  0.877046
eletronicos                                248  1.778226    1.0  0.883502
automotivo                                 369  1.769648    1.0  0.871375
construcao_ferramentas_construcao           69  1.739130    1.0  0.868606
perfumaria                                 233  1.725322    1.0  0.881859
informatica_acessorios                     795  1.708176    1.0  0.866605
eletrodomesticos                            84  1.630952    1.0  0.875087
eletroportateis                             54  1.518519    1.0  0.770708

### (4.2) En kötü ürün kategorileri

👎 *Ortalama değerlendirme puanı* açısından en kötü beş kategoriyi `worst_products` adlı bir değişkene kaydedin.

In [29]:
worst_products = product_categories.tail(5).sort_values(by = [("review_score", "count")],
                                                       ascending = False)
worst_products

review_score                           
                                         count      mean median       std
product_category_name                                                    
informatica_acessorios                     795  1.708176    1.0  0.866605
perfumaria                                 233  1.725322    1.0  0.881859
eletrodomesticos                            84  1.630952    1.0  0.875087
construcao_ferramentas_construcao           69  1.739130    1.0  0.868606
eletroportateis                             54  1.518519    1.0  0.770708

👇 Yalnızca `worst_products` öğelerini içeren bir `worst_products_review` DataFrame oluşturun.

In [30]:
worst_products_reviews = df[df.product_category_name.isin(worst_products.index)]
worst_products_reviews[["review_id",
                        "review_score",
                        "product_category_name",
                        "full_review_cleaned",
                        "most_important_topic",
                        "most_important_words"]
      ]

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
5,ff722b4c68783a4459a3adb9bb4e1d0d,3,informatica_acessorios,produto chegou pc nao conseguiu reconhecer por...,0,"[nao, bom, produto, entrega, gostei, prazo, na..."
14,5f938e5f5f2e9a75710b54feeb9ea610,1,eletrodomesticos,medio peca nao serviu,0,"[nao, bom, produto, entrega, gostei, prazo, na..."
24,b736ff4204044e49e39062584d06fa74,1,eletroportateis,loja anuncia produto entrega outro,1,"[recebi, comprei, nao, so, entregue, apenas, p..."
29,6f885e2fd69412264d9e74e805175d53,1,informatica_acessorios,recebi bem rapido porem veio outro produto nao...,0,"[nao, bom, produto, entrega, gostei, prazo, na..."
34,3db041dd46e9935752c691a2093799c2,1,informatica_acessorios,produto nao recomendo nao ler nenhum dispositivo,0,"[nao, bom, produto, entrega, gostei, prazo, na..."
...,...,...,...,...,...,...
9603,44380245bf1a49875542ea0e7f216986,1,informatica_acessorios,nao recebi produto aind assim receber produto ...,0,"[nao, bom, produto, entrega, gostei, prazo, na..."
9611,970ae7c9df61910a0a30c38670db7ae5,3,informatica_acessorios,probleminha pasta multiuso pra notebook pedi c...,0,"[nao, bom, produto, entrega, gostei, prazo, na..."
9613,504403288b387de1c95a193909ba5dfd,3,perfumaria,chegou bem rapidinho produto razoavel,0,"[nao, bom, produto, entrega, gostei, prazo, na..."
9624,5085bc489aa6b58a29c4f922d59ff826,2,construcao_ferramentas_construcao,problema entrega correio colocou site entregue...,1,"[recebi, comprei, nao, so, entregue, apenas, p..."


### (4.3). En kötü ürünler için konular

❓ En kötü ürünlerin konuları nelerdir? ❓

In [31]:
worst_products_reviews["most_important_topic"].value_counts()

most_important_topic
1    430
2    408
0    397
Name: count, dtype: int64

In [32]:
bad_frequency = list(worst_products_reviews["most_important_topic"].value_counts().index)
bad_frequency

[1, 2, 0]

In [33]:
[topic_word_mixture[i] for i in bad_frequency]

[['recebi',
  'comprei',
  'nao',
  'so',
  'entregue',
  'apenas',
  'produto',
  'pedido',
  'chegou',
  'dois'],
 ['produto',
  'veio',
  'nao',
  'recomendo',
  'diferente',
  'produto veio',
  'qualidade',
  'errado',
  'bem',
  'ruim'],
 ['nao',
  'bom',
  'produto',
  'entrega',
  'gostei',
  'prazo',
  'nao recebi',
  'recebi',
  'ainda',
  'recebi produto']]

## (5) 🎁 Satıcılar...

* En kötü satıcılar tarafından ne tür ürünler satıldı?
* En kötü satıcılar için başlıca yorumlar nelerdir?

### (5.1) En kötü satıcılar

In [38]:
import sys

sys.path.append("/home/mustafa/code/mengMustafaCan/sprint-12")

from olist.seller import Seller
from olist.product import Product
from olist.data import Olist

data = Olist().get_data()
sellers = Seller().get_training_data()
sellers.columns

Index(['seller_id', 'seller_city', 'seller_state', 'delay_to_carrier',
       'wait_time', 'date_first_sale', 'date_last_sale', 'months_on_olist',
       'n_orders', 'quantity', 'quantity_per_order', 'sales'],
      dtype='object')

👇 En kötü satan 10 ürünü seçin ve bunları `worst_sellers` adlı bir değişkene kaydedin.

In [39]:
worst_sellers = sellers[["seller_id", "review_score", "profits"]].sort_values(
    by = "profits",
    ascending = True).head(10)
worst_sellers

KeyError: "['review_score', 'profits'] not in index"

### (5.2) En kötü satıcılar tarafından satılan ürünler

In [40]:
products = Product().get_training_data() [["product_id", "category"]]
products

,product_id,category
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,art
2,96bd76ec8810374ed1b65e291975717f,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,baby
4,9dc1a7de274444849c219cff195d0b71,housewares
...,...,...
31479,a0b7d5a992ccda646f2d34e418fff5a0,furniture_decor
31480,bf4538d88321d0fd4412a93c974510e6,construction_tools_lights
31481,9a7c6041fa9592d9d9ef6cfe62a71f8c,bed_bath_table
31482,83808703fc0706a22e264b9d75f04a2e,computers_accessories


❓ En kötü satıcılar tarafından satılan ürün türleri nelerdir? ❓

In [41]:
sellers_product_category = data["order_items"].merge(products,
                                             on = "product_id", how = "left")[["seller_id", "category"]]

sellers_product_category

,seller_id,category
0,48436dade18ac8b2bce089ec2a041202,cool_stuff
1,dd7ddc04e1b6c2c614352b383efe2d36,pet_shop
2,5b51032eddd242adc84c38acab88f23d,furniture_decor
3,9d7a1d34a5052409006425275ba1c2b4,perfumery
4,df560393f3a51e74553ab94004ba5c87,garden_tools
...,...,...
112645,b8bc237ba3788b23da09c0f1f3a3288c,housewares
112646,f3c38ab652836d21de61fb8314b69182,computers_accessories
112647,c3cfdc648177fdbbbb35635a37472c53,sports_leisure
112648,2b3e4a2a3ea8e01938cabda2a3e5cc79,computers_accessories


In [42]:
sellers_product_category.groupby("seller_id").count()

,category
seller_id,
0015a82c2db000af6aaaf3ae2ecb0532,3
001cca7ae9ae17fb1caed9dfb1094831,239
001e6ad469a905060d959994f1b41e4f,0
002100f778ceb8431b7a1020ff7ab48f,55
003554e2dce176b5555353e4f3555ac8,0
...,...
ffcfefa19b08742c5d315f2791395ee5,0
ffdd9f82b9a447f6f8d4b91554cc7dd3,20
ffeee66ac5d5a62fe688b9d26f83f534,14


### (5.3) En kötü satıcılar için kategoriler ve konular

🎁 İşte bazı kullanışlı işlevler:
- Bir satıcı tarafından satılan ürün kategorilerini göstermek için `focus_seller(seller_id)`
- Bir satıcı için en sık kullanılan konuların en popüler kelimelerini göstermek için `bad_reviews_seller`

In [43]:
def focus_seller(seller_id):
    return sellers_product_category[sellers_product_category.seller_id == seller_id].value_counts()

In [44]:
bad_reviews_sellers = worst_products_reviews.merge(data["order_items"])
bad_reviews_sellers.head(3)

,review_id,length_review,review_score,order_id,product_category_name,full_review,full_review_cleaned,most_important_topic,most_important_words,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,ff722b4c68783a4459a3adb9bb4e1d0d,66,3,b8702f50a689621ab21ef200b55a5b8d,informatica_acessorios,"Produto chegou, mas meu PC não conseguiu reco...",produto chegou pc nao conseguiu reconhecer por...,0,"[nao, bom, produto, entrega, gostei, prazo, na...",1,e76438822c17316d276c27e52e588931,5656537e588803a555b8eb41f07a944b,2018-01-10 22:07:28,24.89,17.63
1,5f938e5f5f2e9a75710b54feeb9ea610,17,1,d9ff0185a300435402f043124f530274,eletrodomesticos,Médio A peça não serviu,medio peca nao serviu,0,"[nao, bom, produto, entrega, gostei, prazo, na...",1,082175ee534338e4fd967076a6f848e5,d91fb3b7d041e83b64a00a3edfb37e4f,2018-05-10 16:54:16,52.00,18.24
2,b736ff4204044e49e39062584d06fa74,43,1,993da8482e658ec7f1d0c6d955604936,eletroportateis,A loja anuncia um produto e entrega outro.,loja anuncia produto entrega outro,1,"[recebi, comprei, nao, so, entregue, apenas, p...",1,c5e19a898511a0e8b60bd98fbc1d72eb,2d2c62bef4c8ba0dd74e0e228e808539,2018-08-21 03:35:16,220.00,38.67


In [45]:
def bad_reviews_seller(bad_reviews_sellers, seller_id):
    mask = (bad_reviews_sellers.seller_id == seller_id)
    temp = bad_reviews_sellers[mask]
    if len(temp) > 0: # satıcı kötü yorumlar veri çerçevesinde görünüyorsa
        most_frequent_topic_seller = list(temp.most_important_topic.value_counts().head(1).index)[0]
        return topic_word_mixture[most_frequent_topic_seller]

❓Bu en az satan ürünlerin her biri için en sık kullanılan ürün kategorilerini ve kelimeleri gösterin ❓

In [46]:
for worst_seller in worst_sellers["seller_id"]:
    print("-"*50)
    print(f"Focusing on the seller #{worst_seller}...")
    print(focus_seller(worst_seller))
    print(bad_reviews_seller(bad_reviews_sellers, worst_seller))


NameError: name 'worst_sellers' is not defined

🏁 Tebrikler. NLP'nin bazı temellerini (Ön İşleme + Vektörleştirme + NB/LDA) öğrendiniz ve bu yeni “uzmanlığı” Karar Bilimi ile birleştirdik.

💾 `git add / commit / push` yapmayı unutmayın.